# 🎙️ AI Voiceover Studio su Colab

Webapp completa (script → voce → video, tab **AI Video** e **Clip studio**) che gira su Google Colab.

**Come si usa:**
1. Aggiungi il secret `OPENROUTER_API_KEY` (icona 🔑 a sinistra → *Secrets*). Serve solo per i tab AI (AI Video, Clip studio, selezione clip, illustrazioni): il tab **Voiceover** con le voci Edge gratuite funziona senza nessuna chiave.
2. Esegui le celle in ordine. La cella 4 apre la webUI con un link pubblico (`*.gradio.live`) da aprire nel browser.
3. I file generati (video, clip, illustrazioni) finiscono in `content/studio_out/<sessione>/`: scaricali dal pannello File a sinistra.

> Nota: YouTube su Colab spesso chiede la verifica anti-bot — usa la cella dei cookies qui sotto (opzionale) oppure cerca su **Wikimedia Commons** (keyless, nessun bot-check).


In [ ]:
# 1) Installazione (~1 minuto)
%%bash
set -e
ffmpeg -version | head -1
pip install -q "ai-voiceover-studio[webui,edge,clip] @ git+https://github.com/mnatali74/ai-voiceover-studio.git@vibe/web-studio-b186e9"
# deno: runtime JavaScript richiesto da yt-dlp per le sfide anti-bot di YouTube
if ! command -v deno >/dev/null 2>&1; then
  curl -fsSL https://deno.land/install.sh | sh -s -- -y >/dev/null 2>&1
fi
echo "Installazione completata."


In [ ]:
# 2) Chiavi API (dai Secrets di Colab, icona 🔑 a sinistra)
import os

try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    print("OPENROUTER_API_KEY: ok")
except Exception:
    if not os.environ.get("OPENROUTER_API_KEY"):
        print("Secret OPENROUTER_API_KEY non trovato: i tab AI non funzioneranno.")
        print("(Voiceover, Quick line e Voices funzionano comunque, sono gratuiti.)")

try:
    from google.colab import userdata
    os.environ["VOICEOVER_PIXABAY_KEY"] = userdata.get("PIXABAY_KEY")
    print("PIXABAY_KEY: ok")
except Exception:
    pass  # opzionale: solo per la fonte Pixabay nel Clip studio


### (Opzionale) Cookies di YouTube

Se YouTube risponde *"Sign in to confirm you're not a bot"*, carica il tuo `cookies.txt`
(export con l'estensione *Get cookies.txt LOCALLY* mentre sei loggato su youtube.com)
nel pannello File di Colab (cartella `/content/`) e ri-esegui la cella qui sotto.
**Il file contiene la tua sessione Google: non condividerlo e non committarlo.**


In [ ]:
# 3) Cookies di YouTube (opzionale)
import os

if os.path.exists("/content/cookies.txt"):
    os.environ["VOICEOVER_YTDLP_COOKIES"] = "/content/cookies.txt"
    print("Cookies caricati da /content/cookies.txt")
else:
    print("Nessun cookies.txt in /content: ok, servono solo se YouTube blocca i download.")
    print("Oppure incolla i link direttamente nel campo 'Or paste YouTube links manually'.")


In [ ]:
# 4) Avvia lo studio — apri il link *.gradio.live che compare qui sotto
from pathlib import Path

from voiceover_studio.webui import launch

launch(workdir=Path("studio_out"), share=True)  # interrompi la cella per fermare il server


In [ ]:
# 5) (Opzionale) Scarica i risultati come zip
import shutil
from pathlib import Path

out = Path("studio_out")
videos = sorted(out.rglob("*.mp4")) + sorted(out.rglob("*.wav"))
print("\n".join(str(v) for v in videos[:40]) or "Nessun file ancora.")
if any(out.iterdir()):
    shutil.make_archive("studio_results", "zip", out)
    print("Creato studio_results.zip: scaricalo dal pannello File a sinistra.")
